In [15]:
# Mount Google Drive (if using Colab)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except:
    IN_COLAB = False
    print("Not running in Colab")

Mounted at /content/drive


# [GPT Link](https://chatgpt.com/share/69b327a6-da94-8005-80c3-f7f09081bc40)

## [Word File](https://docs.google.com/document/d/1ufwWzABa8MhAhmbVzq7El8mLDy3Ra8m-l1jWfWW9ZRE/edit?usp=sharing)

# [Policy_en CSV Link](https://drive.google.com/file/d/17ShQTtD0Ryy6wycrrLfkJ3Ak-80nkp-v/view?usp=sharing)

In [16]:
import pandas as pd
Policy_Csv = "/content/Policy_en.csv"
df = pd.read_csv(Policy_Csv)
df

,policy_id,section,policy_name,rule_id,rule_description,threshold_value,condition,approval_required,risk_level
0,1,Financial,Hardware Purchase Policy,1,Purchases up to $1500 are auto-approved,1500.0,amount <= 1500,No,Medium
1,1,Financial,Hardware Purchase Policy,2,Purchases between $1501–$3000 require manager ...,3000.0,1500 < amount <= 3000,Manager,Medium
2,1,Financial,Hardware Purchase Policy,3,"Purchases above $3000 require manager, finance...",3000.0,amount > 3000,Manager+Finance+CTO,Medium
3,1,Financial,Hardware Purchase Policy,4,Annual hardware budget per employee is $2500,2500.0,annual_budget_limit,No,Medium
4,1,Financial,Hardware Purchase Policy,5,Vendor must be selected from approved vendor list,NaN,vendor_approved == true,No,Medium
5,2,Financial,Travel Expense Policy,1,Economy class required for flights under 6 hours,6.0,flight_hours <= 6,No,Medium
6,2,Financial,Travel Expense Policy,2,Business class allowed for flights over 6 hour...,6.0,flight_hours > 6 AND role == executive,Manager,Medium
7,2,Financial,Travel Expense Policy,3,Hotel cost must not exceed $250 regional,250.0,region == regional,No,Medium
8,2,Financial,Travel Expense Policy,4,Hotel cost must not exceed $400 international ...,400.0,region == international,No,Medium
9,2,Financial,Travel Expense Policy,5,Receipts mandatory for reimbursement,NaN,receipt_provided == true,No,Medium


In [17]:
df= df.drop(columns=["policy_id","rule_id"])
df["full_text"] = df.astype(str).agg(" | ".join, axis=1)
df.head()

,section,policy_name,rule_description,threshold_value,condition,approval_required,risk_level,full_text
0,Financial,Hardware Purchase Policy,Purchases up to $1500 are auto-approved,1500.0,amount <= 1500,No,Medium,Financial | Hardware Purchase Policy | Purchas...
1,Financial,Hardware Purchase Policy,Purchases between $1501–$3000 require manager ...,3000.0,1500 < amount <= 3000,Manager,Medium,Financial | Hardware Purchase Policy | Purchas...
2,Financial,Hardware Purchase Policy,"Purchases above $3000 require manager, finance...",3000.0,amount > 3000,Manager+Finance+CTO,Medium,Financial | Hardware Purchase Policy | Purchas...
3,Financial,Hardware Purchase Policy,Annual hardware budget per employee is $2500,2500.0,annual_budget_limit,No,Medium,Financial | Hardware Purchase Policy | Annual ...
4,Financial,Hardware Purchase Policy,Vendor must be selected from approved vendor list,NaN,vendor_approved == true,No,Medium,Financial | Hardware Purchase Policy | Vendor ...


In [18]:
df["full_text"][0]

'Financial | Hardware Purchase Policy | Purchases up to $1500 are auto-approved | 1500.0 | amount <= 1500 | No | Medium'

In [19]:
# Target
#1) Matching row with userQ  ( Semantic Search)
#2) Matching rule with condition col. (keyword Search)

In [20]:
#  Semantic Search

# 1) Embedding


from sentence_transformers import SentenceTransformer
import numpy as np

embedder = SentenceTransformer("all-MiniLM-L6-v2")
policy_texts = df["full_text"].tolist()

policy_embeddings = embedder.encode(
    policy_texts,
    normalize_embeddings=True
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [21]:
policy_texts[0]

'Financial | Hardware Purchase Policy | Purchases up to $1500 are auto-approved | 1500.0 | amount <= 1500 | No | Medium'

In [22]:
policy_embeddings[0]

array([-1.37062324e-02,  4.47237119e-02, -1.61636863e-02, -5.36067486e-02,
       -1.91186694e-03,  2.49939486e-02,  2.33572014e-02,  4.44731116e-02,
       -4.84552495e-02,  2.69257240e-02,  6.35468587e-02, -1.86100565e-02,
        6.56952197e-03, -5.27026318e-02, -8.51291716e-02, -1.72419846e-02,
        5.09684831e-02, -1.25831231e-01, -2.87467185e-02,  4.32612794e-03,
        2.74031051e-02, -3.38292532e-02, -1.66925248e-02,  1.64940804e-02,
        3.45937572e-02,  6.16535731e-03,  5.32277301e-02,  2.17648856e-02,
       -7.00201327e-03,  3.58746387e-02, -1.00474589e-01, -3.03702261e-02,
        9.29181576e-02, -5.24217449e-02,  6.00387380e-02, -7.98505321e-02,
       -6.13257214e-02, -1.00962847e-01, -3.90885808e-02, -1.18482880e-01,
        1.34699177e-02, -8.85257795e-02, -8.76408592e-02,  3.28780189e-02,
        6.45360947e-02,  2.00531501e-02, -6.23311810e-02,  6.48792833e-02,
        2.42654444e-03,  8.24130923e-02,  4.28125123e-03,  7.37885535e-02,
        4.21453007e-02, -

In [23]:
userQ =  "i need left form my job in 3 Days"
userQ = "I need buy device with 4500$"

userQ_embedding = embedder.encode(
    [userQ],
    normalize_embeddings=True
)

In [24]:
# userQ_embedding  and policy_embeddings

In [25]:
score = (  userQ_embedding @ policy_embeddings.T).squeeze()

# Cosine Similarity apply with normalize_embeddings=True in embedder.encode()
# score = (userQ_embedding @ policy_embeddings.T).squeeze() / (
  #  np.linalg.norm(userQ_embedding) *
 #   np.linalg.norm(policy_embeddings, axis=1)
# )

top_k = 5
top_idx = np.argsort(-score)[:top_k]

for i in top_idx:
    print("Score:", round(score[i], 3))
    print("Text :", df.iloc[i]["full_text"])
    print("-"*60)

Score: 0.478
Text : Financial | Hardware Purchase Policy | Purchases between $1501–$3000 require manager approval and 2 quotes | 3000.0 | 1500 < amount <= 3000 | Manager | Medium
------------------------------------------------------------
Score: 0.369
Text : Financial | Hardware Purchase Policy | Purchases up to $1500 are auto-approved | 1500.0 | amount <= 1500 | No | Medium
------------------------------------------------------------
Score: 0.357
Text : Financial | Hardware Purchase Policy | Purchases above $3000 require manager, finance, and CTO approval | 3000.0 | amount > 3000 | Manager+Finance+CTO | Medium
------------------------------------------------------------
Score: 0.311
Text : Financial | Hardware Purchase Policy | Annual hardware budget per employee is $2500 | 2500.0 | annual_budget_limit | No | Medium
------------------------------------------------------------
Score: 0.263
Text : Financial | Hardware Purchase Policy | Vendor must be selected from approved vendor list 

In [26]:
def retrive_cand(userQ,policy_embeddings):
  out = []
  userQ_embedding = embedder.encode(
    [userQ],
    normalize_embeddings=True
  )
  score = (  userQ_embedding @ policy_embeddings.T).squeeze()
  top_k = 5
  top_idx = np.argsort(-score)[:top_k]
  for i in top_idx:
    #  print("Score:", round(score[i], 3))

    out.append({ "Score":round(score[i], 3)  ,  "full_text":df.iloc[i]["full_text"]})
     # print("Text :", df.iloc[i]["full_text"])
    #  print("-"*60)
  return out

# 2) Matching rule with condition col. (keyword Search)

In [27]:
userQ =  "i need left form my job in 3 Days"
retrive_cand(userQ,policy_embeddings)

[{'Score': np.float32(0.241),
  'full_text': 'Legal | Contract Termination Policy | Contracts must include minimum 30-day termination notice | 30.0 | termination_notice_days >= 30 | Legal | High'},
 {'Score': np.float32(0.239),
  'full_text': 'HR | Remote Work Equipment Policy | Replacement allowed every 3 years | 3.0 | years_since_last_replacement >= 3 | Manager | Medium'},
 {'Score': np.float32(0.161),
  'full_text': 'Legal | Contract Termination Policy | Indemnification clauses must be reviewed by legal | nan | indemnification_clause == true | Legal | High'},
 {'Score': np.float32(0.158),
  'full_text': 'Legal | Vendor Agreement Policy | Payment terms must not exceed Net 60 unless CFO approval | 60.0 | payment_terms_days <= 60 | CFO_if_exceeds | High'},
 {'Score': np.float32(0.152),
  'full_text': 'Legal | Contract Termination Policy | Liability cap must not exceed 12 months of contract value | 12.0 | liability_cap_months <= 12 | Legal | High'}]

In [28]:
userQ = "I need buy device with 4500$"
top5Cand = retrive_cand(userQ,policy_embeddings)
top5Cand

[{'Score': np.float32(0.478),
  'full_text': 'Financial | Hardware Purchase Policy | Purchases between $1501–$3000 require manager approval and 2 quotes | 3000.0 | 1500 < amount <= 3000 | Manager | Medium'},
 {'Score': np.float32(0.369),
  'full_text': 'Financial | Hardware Purchase Policy | Purchases up to $1500 are auto-approved | 1500.0 | amount <= 1500 | No | Medium'},
 {'Score': np.float32(0.357),
  'full_text': 'Financial | Hardware Purchase Policy | Purchases above $3000 require manager, finance, and CTO approval | 3000.0 | amount > 3000 | Manager+Finance+CTO | Medium'},
 {'Score': np.float32(0.311),
  'full_text': 'Financial | Hardware Purchase Policy | Annual hardware budget per employee is $2500 | 2500.0 | annual_budget_limit | No | Medium'},
 {'Score': np.float32(0.263),
  'full_text': 'Financial | Hardware Purchase Policy | Vendor must be selected from approved vendor list | nan | vendor_approved == true | No | Medium'}]

In [29]:
# 2) Matching rule with condition col. (keyword Search)

# A) extract number form userQ
# B) extract cond from full text in top5 cand
# C) check cond ???

In [30]:
import re

def extract_amount(text):
    match = re.search(r'[\d,]+', text)
    if match:
        return int(match.group().replace(",", ""))
    return None

amount_userQ = extract_amount(userQ)
amount_userQ

4500

In [31]:
def fullsystem(userQ,policy_embeddings):
  out = []
  amount_userQ = extract_amount(userQ) # 1

  userQ_embedding = embedder.encode(
    [userQ],
    normalize_embeddings=True
  )
  score = (  userQ_embedding @ policy_embeddings.T).squeeze()
  top_k = 5
  top_idx = np.argsort(-score)[:top_k]
  num = 0
  matched = []
  for i in top_idx:
   # print(f"----------- top {num+1} -----------")
    condition = df.iloc[i]["condition"]
    #print(condition)
    expr= condition.replace("amount",str(amount_userQ))
    #print(expr)
    try:
      if eval(expr) == True:
        matched.append({ "Score":round(score[i], 3)  ,  "full_text":df.iloc[i]["full_text"]})

    except:
      pass
    num +=1

  return matched


In [32]:
userQ = "I need buy device with 4500$"

fullsystem(userQ,policy_embeddings)

[{'Score': np.float32(0.357),
  'full_text': 'Financial | Hardware Purchase Policy | Purchases above $3000 require manager, finance, and CTO approval | 3000.0 | amount > 3000 | Manager+Finance+CTO | Medium'}]

In [33]:
userQ = "I need buy device with 1000$"

fullsystem(userQ,policy_embeddings)

[{'Score': np.float32(0.34),
  'full_text': 'Financial | Hardware Purchase Policy | Purchases up to $1500 are auto-approved | 1500.0 | amount <= 1500 | No | Medium'}]

In [34]:
userQ = "I need buy device with 2500$"

fullsystem(userQ,policy_embeddings)

[{'Score': np.float32(0.437),
  'full_text': 'Financial | Hardware Purchase Policy | Purchases between $1501–$3000 require manager approval and 2 quotes | 3000.0 | 1500 < amount <= 3000 | Manager | Medium'}]

# 3) Hybrid Retrieval (Semantic + BM25)

In [35]:
!pip install rank-bm25

In [36]:
# ---------------------------------
# 3) Hybrid Retrieval (Semantic + BM25)
# ---------------------------------

# install library (run once)
# !pip install rank-bm25

from rank_bm25 import BM25Okapi
import re
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
import nltk

nltk.download("stopwords")

stop_words = set(stopwords.words("english"))
stemmer = PorterStemmer()

def simple_tokenize(text):

    # lower
    text = text.lower()

    # keep numbers
    tokens = re.findall(r"[a-zA-Z0-9]+", text)

    # remove stopwords
    tokens = [t for t in tokens if t not in stop_words]

    # stemming
    tokens = [stemmer.stem(t) for t in tokens]

    return tokens
# build BM25 index from policy texts
tokenized_corpus = [simple_tokenize(text) for text in policy_texts]

bm25 = BM25Okapi(tokenized_corpus)

print("BM25 index ready")

BM25 index ready


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [37]:
def retrieve_bm25(userQ, policy_texts, bm25, top_k=5):

    query_tokens = simple_tokenize(userQ)

    scores = bm25.get_scores(query_tokens)

    top_idx = np.argsort(-scores)[:top_k]

    results = []

    for i in top_idx:

        results.append({
            "Score": round(scores[i],3),
            "full_text": policy_texts[i],
            "method":"bm25"
        })

    return results

In [38]:
userQ = "I need buy device with 4500$"
retrieve_bm25(userQ, policy_texts, bm25, top_k=5)

[{'Score': np.float64(0.0),
  'full_text': 'Financial | Hardware Purchase Policy | Purchases up to $1500 are auto-approved | 1500.0 | amount <= 1500 | No | Medium',
  'method': 'bm25'},
 {'Score': np.float64(0.0),
  'full_text': 'Financial | Hardware Purchase Policy | Purchases between $1501–$3000 require manager approval and 2 quotes | 3000.0 | 1500 < amount <= 3000 | Manager | Medium',
  'method': 'bm25'},
 {'Score': np.float64(0.0),
  'full_text': 'Financial | Hardware Purchase Policy | Purchases above $3000 require manager, finance, and CTO approval | 3000.0 | amount > 3000 | Manager+Finance+CTO | Medium',
  'method': 'bm25'},
 {'Score': np.float64(0.0),
  'full_text': 'Financial | Hardware Purchase Policy | Annual hardware budget per employee is $2500 | 2500.0 | annual_budget_limit | No | Medium',
  'method': 'bm25'},
 {'Score': np.float64(0.0),
  'full_text': 'Financial | Hardware Purchase Policy | Vendor must be selected from approved vendor list | nan | vendor_approved == true 

In [39]:
def normalize_scores(results):

    scores = [r["Score"] for r in results]

    min_s = min(scores)
    max_s = max(scores)

    for r in results:

        if max_s == min_s:
            r["norm_score"] = 1.0
        else:
            r["norm_score"] = (r["Score"] - min_s) / (max_s - min_s)

    return results

In [40]:
def hybrid_retrieve(userQ, policy_embeddings, policy_texts, bm25):

    semantic_results = retrive_cand(userQ, policy_embeddings)
    bm25_results = retrieve_bm25(userQ, policy_texts, bm25)

    semantic_results = normalize_scores(semantic_results)
    bm25_results = normalize_scores(bm25_results)

    combined = {}

    # semantic results
    for r in semantic_results:

        text = r["full_text"]

        combined[text] = {
            "full_text": text,
            "semantic": r["norm_score"],
            "bm25": 0
        }

    # bm25 results
    for r in bm25_results:

        text = r["full_text"]

        if text not in combined:

            combined[text] = {
                "full_text": text,
                "semantic": 0,
                "bm25": r["norm_score"]
            }

        else:

            combined[text]["bm25"] = r["norm_score"]

    final = []

    for item in combined.values():
## final_score = 0.7 * semantic_score + 0.3 * bm25_score

        score = (
            0.75 * item["semantic"] +
            0.25 * item["bm25"]
        )

        final.append({
            "Score": round(score,3),
            "full_text": item["full_text"]
        })

    final = sorted(final, key=lambda x: x["Score"], reverse=True)

    return final[:5]

In [41]:
userQ = "I need buy device with 4000$"

hybrid_top = hybrid_retrieve(
    userQ,
    policy_embeddings,
    policy_texts,
    bm25
)

for r in hybrid_top:

    print("Score:", r["Score"])
    print("Text :", r["full_text"])
    print("-"*60)

Score: 1.0
Text : Financial | Hardware Purchase Policy | Purchases between $1501–$3000 require manager approval and 2 quotes | 3000.0 | 1500 < amount <= 3000 | Manager | Medium
------------------------------------------------------------
Score: 0.705
Text : Financial | Hardware Purchase Policy | Purchases up to $1500 are auto-approved | 1500.0 | amount <= 1500 | No | Medium
------------------------------------------------------------
Score: 0.673
Text : Financial | Hardware Purchase Policy | Purchases above $3000 require manager, finance, and CTO approval | 3000.0 | amount > 3000 | Manager+Finance+CTO | Medium
------------------------------------------------------------
Score: 0.509
Text : Financial | Hardware Purchase Policy | Annual hardware budget per employee is $2500 | 2500.0 | annual_budget_limit | No | Medium
------------------------------------------------------------
Score: 0.25
Text : Financial | Hardware Purchase Policy | Vendor must be selected from approved vendor list | n

# Tasks — Building a Practical RAG Decision System

In this lab you will extend the **Policy RAG system** step by step.

The system currently can:

* convert policies to embeddings
* retrieve **Top‑5 policies using semantic search**
* check **numeric rules**

Now you will extend the system to make it closer to a **real AI decision system**.

---

## Task 1 — Compare Retrieval Methods

First, test the difference between **Semantic Search** and **BM25 Search** VS ** Matching rule with condition col**.

Run the same user query using both retrieval methods.

Example query:

```
I need to buy a device for 3500$
```

Run:

* semantic retrieval
* Matching rule with condition col
* BM25 retrieval

### Experiment

Observe the results:

* Which policies appear in **semantic search**?
* Which policies appear in **Matching rule with condition col**?
* Which policies appear in **BM25**?

Try more queries:

```
buy device 3500
hardware purchase
vendor agreement
purchase approval
```

---

In [ ]:
user_query = "I need to buy a device for 3500$"

print(f"User Query: {user_query}")
print("\n--- Semantic Retrieval ---")
semantic_results = retrive_cand(user_query, policy_embeddings)
for res in semantic_results:
    print(f"Score: {res['Score']}\nText: {res['full_text']}\n")

print("\n--- Matching Rule with Condition Column ---")
matched_rules = fullsystem(user_query, policy_embeddings)
if matched_rules:
    for res in matched_rules:
        print(f"Score: {res['Score']}\nText: {res['full_text']}\n")
else:
    print("No exact match found based on condition.")

print("\n--- BM25 Retrieval ---")
bm25_results = retrieve_bm25(user_query, policy_texts, bm25)
for res in bm25_results:
    print(f"Score: {res['Score']}\nText: {res['full_text']}\n")

print("\n--- Matching Rule with BM25 Retrieval (Condition Filtered) ---")
# The fullsystem function is not designed to filter BM25 results directly using embeddings.
# Instead, we will manually apply the condition check to the BM25 retrieved policies.

bm25_matched_rules = []
amount_userQ_bm25 = extract_amount(user_query) # Extract amount from the user query

if amount_userQ_bm25 is not None:
    for res_bm25 in bm25_results:
        # Find the original row in df that corresponds to this full_text
        # This is a lookup, as bm25_results only provides full_text and score
        matching_df_row = df[df['full_text'] == res_bm25['full_text']]
        if not matching_df_row.empty:
            condition_bm25 = matching_df_row.iloc[0]['condition']
            expr_bm25 = condition_bm25.replace("amount", str(amount_userQ_bm25))
            try:
                if eval(expr_bm25) == True:
                    bm25_matched_rules.append(res_bm25)
            except:
                pass

if bm25_matched_rules:
    for res_bm25 in bm25_matched_rules:
        print(f"Score: {res_bm25['Score']}\nText: {res_bm25['full_text']}\n")
else:
    print("No exact match found based on condition in BM25 results.")

User Query: I need to buy a device for 3500$

--- Semantic Retrieval ---
Score: 0.48399999737739563
Text: Financial | Hardware Purchase Policy | Purchases between $1501–$3000 require manager approval and 2 quotes | 3000.0 | 1500 < amount <= 3000 | Manager | Medium

Score: 0.42500001192092896
Text: Financial | Hardware Purchase Policy | Purchases up to $1500 are auto-approved | 1500.0 | amount <= 1500 | No | Medium

Score: 0.37299999594688416
Text: Financial | Hardware Purchase Policy | Purchases above $3000 require manager, finance, and CTO approval | 3000.0 | amount > 3000 | Manager+Finance+CTO | Medium

Score: 0.35100001096725464
Text: Financial | Hardware Purchase Policy | Annual hardware budget per employee is $2500 | 2500.0 | annual_budget_limit | No | Medium

Score: 0.25600001215934753
Text: Financial | Hardware Purchase Policy | Vendor must be selected from approved vendor list | nan | vendor_approved == true | No | Medium


--- Matching Rule with Condition Column ---
Score: 0.3

In [43]:
user_query = "buy device 3500"

print(f"User Query: {user_query}")
print("\n--- Semantic Retrieval ---")
semantic_results = retrive_cand(user_query, policy_embeddings)
for res in semantic_results:
    print(f"Score: {res['Score']}\nText: {res['full_text']}\n")

print("\n--- Matching Rule with Condition Column ---")
matched_rules = fullsystem(user_query, policy_embeddings)
if matched_rules:
    for res in matched_rules:
        print(f"Score: {res['Score']}\nText: {res['full_text']}\n")
else:
    print("No exact match found based on condition.")

print("\n--- BM25 Retrieval ---")
bm25_results = retrieve_bm25(user_query, policy_texts, bm25)
for res in bm25_results:
    print(f"Score: {res['Score']}\nText: {res['full_text']}\n")

print("\n--- Matching Rule with BM25 Retrieval (Condition Filtered) ---")
# The fullsystem function is not designed to filter BM25 results directly using embeddings.
# Instead, we will manually apply the condition check to the BM25 retrieved policies.

bm25_matched_rules = []
amount_userQ_bm25 = extract_amount(user_query) # Extract amount from the user query

if amount_userQ_bm25 is not None:
    for res_bm25 in bm25_results:
        # Find the original row in df that corresponds to this full_text
        # This is a lookup, as bm25_results only provides full_text and score
        matching_df_row = df[df['full_text'] == res_bm25['full_text']]
        if not matching_df_row.empty:
            condition_bm25 = matching_df_row.iloc[0]['condition']
            expr_bm25 = condition_bm25.replace("amount", str(amount_userQ_bm25))
            try:
                if eval(expr_bm25) == True:
                    bm25_matched_rules.append(res_bm25)
            except:
                pass

if bm25_matched_rules:
    for res_bm25 in bm25_matched_rules:
        print(f"Score: {res_bm25['Score']}\nText: {res_bm25['full_text']}\n")
else:
    print("No exact match found based on condition in BM25 results.")

User Query: buy device 3500

--- Semantic Retrieval ---
Score: 0.44200000166893005
Text: Financial | Hardware Purchase Policy | Purchases between $1501–$3000 require manager approval and 2 quotes | 3000.0 | 1500 < amount <= 3000 | Manager | Medium

Score: 0.39800000190734863
Text: Financial | Hardware Purchase Policy | Purchases up to $1500 are auto-approved | 1500.0 | amount <= 1500 | No | Medium

Score: 0.3880000114440918
Text: Financial | Hardware Purchase Policy | Purchases above $3000 require manager, finance, and CTO approval | 3000.0 | amount > 3000 | Manager+Finance+CTO | Medium

Score: 0.3619999885559082
Text: Financial | Hardware Purchase Policy | Annual hardware budget per employee is $2500 | 2500.0 | annual_budget_limit | No | Medium

Score: 0.31299999356269836
Text: Financial | Hardware Purchase Policy | Vendor must be selected from approved vendor list | nan | vendor_approved == true | No | Medium


--- Matching Rule with Condition Column ---
Score: 0.3880000114440918
Tex

In [44]:
user_query = "hardware purchase"

print(f"User Query: {user_query}")
print("\n--- Semantic Retrieval ---")
semantic_results = retrive_cand(user_query, policy_embeddings)
for res in semantic_results:
    print(f"Score: {res['Score']}\nText: {res['full_text']}\n")

print("\n--- Matching Rule with Condition Column ---")
matched_rules = fullsystem(user_query, policy_embeddings)
if matched_rules:
    for res in matched_rules:
        print(f"Score: {res['Score']}\nText: {res['full_text']}\n")
else:
    print("No exact match found based on condition.")

print("\n--- BM25 Retrieval ---")
bm25_results = retrieve_bm25(user_query, policy_texts, bm25)
for res in bm25_results:
    print(f"Score: {res['Score']}\nText: {res['full_text']}\n")

print("\n--- Matching Rule with BM25 Retrieval (Condition Filtered) ---")
# The fullsystem function is not designed to filter BM25 results directly using embeddings.
# Instead, we will manually apply the condition check to the BM25 retrieved policies.

bm25_matched_rules = []
amount_userQ_bm25 = extract_amount(user_query) # Extract amount from the user query

if amount_userQ_bm25 is not None:
    for res_bm25 in bm25_results:
        # Find the original row in df that corresponds to this full_text
        # This is a lookup, as bm25_results only provides full_text and score
        matching_df_row = df[df['full_text'] == res_bm25['full_text']]
        if not matching_df_row.empty:
            condition_bm25 = matching_df_row.iloc[0]['condition']
            expr_bm25 = condition_bm25.replace("amount", str(amount_userQ_bm25))
            try:
                if eval(expr_bm25) == True:
                    bm25_matched_rules.append(res_bm25)
            except:
                pass

if bm25_matched_rules:
    for res_bm25 in bm25_matched_rules:
        print(f"Score: {res_bm25['Score']}\nText: {res_bm25['full_text']}\n")
else:
    print("No exact match found based on condition in BM25 results.")

User Query: hardware purchase

--- Semantic Retrieval ---
Score: 0.5649999976158142
Text: Financial | Hardware Purchase Policy | Purchases between $1501–$3000 require manager approval and 2 quotes | 3000.0 | 1500 < amount <= 3000 | Manager | Medium

Score: 0.4740000069141388
Text: Financial | Hardware Purchase Policy | Purchases up to $1500 are auto-approved | 1500.0 | amount <= 1500 | No | Medium

Score: 0.47200000286102295
Text: Financial | Hardware Purchase Policy | Purchases above $3000 require manager, finance, and CTO approval | 3000.0 | amount > 3000 | Manager+Finance+CTO | Medium

Score: 0.39899998903274536
Text: Financial | Hardware Purchase Policy | Annual hardware budget per employee is $2500 | 2500.0 | annual_budget_limit | No | Medium

Score: 0.37299999594688416
Text: Financial | Hardware Purchase Policy | Vendor must be selected from approved vendor list | nan | vendor_approved == true | No | Medium


--- Matching Rule with Condition Column ---
No exact match found based 


## Task 2 — Use the LLM to Re‑Rank Candidates

Now add the **LLM to the retrieval pipeline**.

Instead of trusting the retrieval scores directly, ask the LLM to choose the **best rule from the candidates**.

Input to the LLM:

* the user question
* the retrieved candidate policies

The LLM should return **the single most relevant rule**.

### Example

User question

```
I need to buy a device for 3500$
```

Candidate rules

```
amount <= 1500
1500 < amount <= 3000
amount > 3000
```

Expected LLM output

```
amount > 3000
```

### Experiment

Run the system with different queries and see how the LLM selects rules.

---

In [ ]:
# Import the OpenAI library
from openai import OpenAI
from google.colab import userdata

# Configure the API key
# It's recommended to store your API key securely in Colab secrets
# To do this, go to 'Secrets' tab (🔑 icon) on the left sidebar in Colab,
# click '+ New secret', set name to 'secretName' and paste your OpenAI API key.
# Then, toggle 'Notebook access' on for this notebook.
API_KEY = userdata.get('secretName')
client = OpenAI(
    base_url='https://openrouter.ai/api/v1',  # OpenRouter base URL
    api_key=API_KEY
)

def llm_rerank_candidates(user_question, candidate_policies):
    """
    Uses an LLM to re-rank candidate policies and return the single most relevant rule.

    Args:
        user_question (str): The user's original question.
        candidate_policies (list): A list of dictionaries, where each dictionary
                                   contains 'full_text' of a policy.

    Returns:
        str: The full_text of the single most relevant policy as chosen by the LLM,
             or None if no suitable rule was found or an error occurred.
    """
    if not candidate_policies:
        return None # No candidates to re-rank

    # Format candidate policies for the prompt
    policy_options = []
    for i, policy in enumerate(candidate_policies):
        policy_options.append(f"Option {i+1}: {policy['full_text']}")
    policies_str = "\n".join(policy_options)

    prompt_content = f"""User Question: {user_question}

Here are several policy rules:
{policies_str}

Based on the user's question, identify the SINGLE most relevant policy rule from the options provided. 
Your answer should be ONLY the option number (e.g., 'Option 3'). Do not add any extra text or explanation.

Most relevant option:"""

    try:
        response = client.chat.completions.create(
            model="openai/gpt-3.5-turbo", # Using an OpenRouter compatible model
            messages=[
                {"role": "user", "content": prompt_content}
            ],
            temperature=0.0
        )
        chosen_option_str = response.choices[0].message.content.strip() # e.g., "Option 3"

        import re
        # Extract the number
        match = re.search(r'Option (\d+)', chosen_option_str)
        if match:
            option_number = int(match.group(1))
            if 1 <= option_number <= len(candidate_policies):
                return candidate_policies[option_number - 1]['full_text']
            else:
                print(f"LLM chose an invalid option number: {option_number}")
                return None
        else:
            print(f"LLM did not return a valid option format: {chosen_option_str}")
            return None

    except Exception as e:
        print(f"Error during LLM re-ranking: {e}")
        return None


In [47]:
print("\n--- LLM Re-Ranking Example ---")
user_query = "I need to buy a device for 3500$"
# First, retrieve candidates using semantic search (or hybrid retrieval later)
semantic_candidates = retrive_cand(user_query, policy_embeddings)

if semantic_candidates :
    print(f"User Query: {user_query}")
    print("Candidate policies (from Semantic Retrieval):")
    for candidate in semantic_candidates:
        print(f"- {candidate['full_text']}")

    best_rule_llm = llm_rerank_candidates(user_query, semantic_candidates)
    print(f"\nLLM selected the most relevant rule: {best_rule_llm}")
else:
    print("No candidate policies found for re-ranking.")


--- LLM Re-Ranking Example ---
User Query: I need to buy a device for 3500$
Candidate policies (from Semantic Retrieval):
- Financial | Hardware Purchase Policy | Purchases between $1501–$3000 require manager approval and 2 quotes | 3000.0 | 1500 < amount <= 3000 | Manager | Medium
- Financial | Hardware Purchase Policy | Purchases up to $1500 are auto-approved | 1500.0 | amount <= 1500 | No | Medium
- Financial | Hardware Purchase Policy | Purchases above $3000 require manager, finance, and CTO approval | 3000.0 | amount > 3000 | Manager+Finance+CTO | Medium
- Financial | Hardware Purchase Policy | Annual hardware budget per employee is $2500 | 2500.0 | annual_budget_limit | No | Medium
- Financial | Hardware Purchase Policy | Vendor must be selected from approved vendor list | nan | vendor_approved == true | No | Medium

LLM selected the most relevant rule: Financial | Hardware Purchase Policy | Purchases between $1501–$3000 require manager approval and 2 quotes | 3000.0 | 1500 < am

In [48]:
print("\n--- LLM Re-Ranking Example ---")
user_query = "buy device 3500"
# First, retrieve candidates using semantic search (or hybrid retrieval later)
semantic_candidates = retrive_cand(user_query, policy_embeddings)

if semantic_candidates :
    print(f"User Query: {user_query}")
    print("Candidate policies (from Semantic Retrieval):")
    for candidate in semantic_candidates:
        print(f"- {candidate['full_text']}")

    best_rule_llm = llm_rerank_candidates(user_query, semantic_candidates)
    print(f"\nLLM selected the most relevant rule: {best_rule_llm}")
else:
    print("No candidate policies found for re-ranking.")


--- LLM Re-Ranking Example ---
User Query: buy device 3500
Candidate policies (from Semantic Retrieval):
- Financial | Hardware Purchase Policy | Purchases between $1501–$3000 require manager approval and 2 quotes | 3000.0 | 1500 < amount <= 3000 | Manager | Medium
- Financial | Hardware Purchase Policy | Purchases up to $1500 are auto-approved | 1500.0 | amount <= 1500 | No | Medium
- Financial | Hardware Purchase Policy | Purchases above $3000 require manager, finance, and CTO approval | 3000.0 | amount > 3000 | Manager+Finance+CTO | Medium
- Financial | Hardware Purchase Policy | Annual hardware budget per employee is $2500 | 2500.0 | annual_budget_limit | No | Medium
- Financial | Hardware Purchase Policy | Vendor must be selected from approved vendor list | nan | vendor_approved == true | No | Medium

LLM selected the most relevant rule: Financial | Hardware Purchase Policy | Purchases between $1501–$3000 require manager approval and 2 quotes | 3000.0 | 1500 < amount <= 3000 | Ma

In [56]:
print("\n--- LLM Re-Ranking Example ---")
user_query = "hardware purchase"
# First, retrieve candidates using semantic search (or hybrid retrieval later)
semantic_candidates = retrive_cand(user_query, policy_embeddings)

if semantic_candidates :
    print(f"User Query: {user_query}")
    print("Candidate policies (from Semantic Retrieval):")
    for candidate in semantic_candidates:
        print(f"- {candidate['full_text']}")

    best_rule_llm = llm_rerank_candidates(user_query, semantic_candidates)
    print(f"\nLLM selected the most relevant rule: {best_rule_llm}")
else:
    print("No candidate policies found for re-ranking.")


--- LLM Re-Ranking Example ---
User Query: hardware purchase
Candidate policies (from Semantic Retrieval):
- Financial | Hardware Purchase Policy | Purchases between $1501–$3000 require manager approval and 2 quotes | 3000.0 | 1500 < amount <= 3000 | Manager | Medium
- Financial | Hardware Purchase Policy | Purchases up to $1500 are auto-approved | 1500.0 | amount <= 1500 | No | Medium
- Financial | Hardware Purchase Policy | Purchases above $3000 require manager, finance, and CTO approval | 3000.0 | amount > 3000 | Manager+Finance+CTO | Medium
- Financial | Hardware Purchase Policy | Annual hardware budget per employee is $2500 | 2500.0 | annual_budget_limit | No | Medium
- Financial | Hardware Purchase Policy | Vendor must be selected from approved vendor list | nan | vendor_approved == true | No | Medium

LLM selected the most relevant rule: Financial | Hardware Purchase Policy | Purchases above $3000 require manager, finance, and CTO approval | 3000.0 | amount > 3000 | Manager+Fin

In [ ]:
# Import the OpenAI library
from openai import OpenAI
from google.colab import userdata

# Configure the API key
# It's recommended to store your API key securely in Colab secrets
# To do this, go to 'Secrets' tab (🔑 icon) on the left sidebar in Colab,
# click '+ New secret', set name to 'secretName' and paste your OpenAI API key.
# Then, toggle 'Notebook access' on for this notebook.
API_KEY = userdata.get('secretName')
client = OpenAI(
    base_url='https://openrouter.ai/api/v1',  # OpenRouter base URL
    api_key=API_KEY
)

def llm_rerank_candidates(user_question, candidate_policies):
    """
    Uses an LLM to re-rank candidate policies and return the single most relevant rule.

    Args:
        user_question (str): The user's original question.
        candidate_policies (list): A list of dictionaries, where each dictionary
                                   contains 'full_text' of a policy.

    Returns:
        str: The full_text of the single most relevant policy as chosen by the LLM,
             or None if no suitable rule was found or an error occurred.
    """
    if not candidate_policies:
        return None # No candidates to re-rank

    # Format candidate policies for the prompt
    policy_options = []
    for i, policy in enumerate(candidate_policies):
        policy_options.append(f"Option {i+1}: {policy['full_text']}")
    policies_str = "\n".join(policy_options)

    prompt_content = f"""User Question: {user_question}

Here are several policy rules:
{policies_str}

Based on the user's question, identify the SINGLE most relevant policy rule from the options provided. 
Pay close attention to any numerical values in the user's question (e.g., $3500) and compare them accurately to the numerical conditions (e.g., 'amount > 3000', 'amount <= 1500') in each policy when determining relevance. 
Your answer should be ONLY the option number (e.g., 'Option 3'). Do not add any extra text or explanation.

Most relevant option:"""

    try:
        response = client.chat.completions.create(
            model="openai/gpt-3.5-turbo", # Using an OpenRouter compatible model
            messages=[
                {"role": "user", "content": prompt_content}
            ],
            temperature=0.0
        )
        chosen_option_str = response.choices[0].message.content.strip() # e.g., "Option 3"

        import re
        # Extract the number
        match = re.search(r'Option (\d+)', chosen_option_str)
        if match:
            option_number = int(match.group(1))
            if 1 <= option_number <= len(candidate_policies):
                return candidate_policies[option_number - 1]['full_text']
            else:
                print(f"LLM chose an invalid option number: {option_number}")
                return None
        else:
            print(f"LLM did not return a valid option format: {chosen_option_str}")
            return None

    except Exception as e:
        print(f"Error during LLM re-ranking: {e}")
        return None


In [50]:
print("\n--- LLM Re-Ranking Example ---")
user_query = "hardware purchase"
# First, retrieve candidates using semantic search (or hybrid retrieval later)
semantic_candidates = retrive_cand(user_query, policy_embeddings)

if semantic_candidates :
    print(f"User Query: {user_query}")
    print("Candidate policies (from Semantic Retrieval):")
    for candidate in semantic_candidates:
        print(f"- {candidate['full_text']}")

    best_rule_llm = llm_rerank_candidates(user_query, semantic_candidates)
    print(f"\nLLM selected the most relevant rule: {best_rule_llm}")
else:
    print("No candidate policies found for re-ranking.")


--- LLM Re-Ranking Example ---
User Query: hardware purchase
Candidate policies (from Semantic Retrieval):
- Financial | Hardware Purchase Policy | Purchases between $1501–$3000 require manager approval and 2 quotes | 3000.0 | 1500 < amount <= 3000 | Manager | Medium
- Financial | Hardware Purchase Policy | Purchases up to $1500 are auto-approved | 1500.0 | amount <= 1500 | No | Medium
- Financial | Hardware Purchase Policy | Purchases above $3000 require manager, finance, and CTO approval | 3000.0 | amount > 3000 | Manager+Finance+CTO | Medium
- Financial | Hardware Purchase Policy | Annual hardware budget per employee is $2500 | 2500.0 | annual_budget_limit | No | Medium
- Financial | Hardware Purchase Policy | Vendor must be selected from approved vendor list | nan | vendor_approved == true | No | Medium

LLM selected the most relevant rule: Financial | Hardware Purchase Policy | Purchases above $3000 require manager, finance, and CTO approval | 3000.0 | amount > 3000 | Manager+Fin

In [51]:
for m in genai.list_models():
  if 'generateContent' in m.supported_generation_methods:
    print(m.name)

NameError: name 'genai' is not defined


## Task 3 — Apply Rule Filtering

After the LLM selects a rule candidate, the system must verify that the rule **actually matches the user request**.

Example rule

```
amount > 3000
```

Example request

```
buy device for 3500$
```

The system should check if the condition is **true**.

### Experiment

Test the system with:

```
buy device for 800$
buy device for 1800$
buy device for 3500$
```

Observe how the selected rule changes.

---

In [52]:
def apply_rule_filtering(user_query, llm_selected_rule_text):
    """
    Applies rule filtering to an LLM-selected policy based on the user's query.

    Args:
        user_query (str): The user's original question.
        llm_selected_rule_text (str): The full text of the policy rule selected by the LLM.

    Returns:
        dict or None: The policy dictionary if the condition is met, otherwise None.
                      Includes 'is_match': True/False and 'explanation' keys.
    """
    amount_userQ = extract_amount(user_query)

    # Find the row in the DataFrame corresponding to the LLM-selected rule text
    matching_df_row = df[df['full_text'] == llm_selected_rule_text]

    if matching_df_row.empty:
        return {"is_match": False, "explanation": f"Selected rule not found in policies: {llm_selected_rule_text}"}

    policy_row = matching_df_row.iloc[0]
    condition = policy_row['condition']
    rule_description = policy_row['rule_description']
    policy_name = policy_row['policy_name']
    approval_required = policy_row['approval_required']

    if amount_userQ is not None and isinstance(condition, str) and 'amount' in condition:
        # Replace 'amount' in the condition string with the extracted amount
        expr = condition.replace("amount", str(amount_userQ))
        try:
            if eval(expr):
                return {
                    "is_match": True,
                    "policy_name": policy_name,
                    "rule_description": rule_description,
                    "condition": condition,
                    "approval_required": approval_required,
                    "full_text": llm_selected_rule_text,
                    "explanation": f"The policy '{policy_name}' rule '{rule_description}' applies because the amount ${amount_userQ} meets the condition '{condition}'. Approval is: {approval_required}."
                }
            else:
                return {
                    "is_match": False,
                    "policy_name": policy_name,
                    "rule_description": rule_description,
                    "condition": condition,
                    "full_text": llm_selected_rule_text,
                    "explanation": f"The policy '{policy_name}' rule '{rule_description}' does NOT apply because the amount ${amount_userQ} does not meet the condition '{condition}'."
                }
        except Exception as e:
            return {"is_match": False, "explanation": f"Error evaluating condition '{condition}': {e}"}
    elif amount_userQ is None and isinstance(condition, str) and 'amount' in condition:
        return {"is_match": False, "explanation": f"No amount found in query to evaluate against condition '{condition}'."}
    else:
        # Handle non-amount based conditions or cases where amount is not relevant
        return {
            "is_match": True, # Assume true if no specific amount condition applies or can be evaluated
            "policy_name": policy_name,
            "rule_description": rule_description,
            "condition": condition,
            "approval_required": approval_required,
            "full_text": llm_selected_rule_text,
            "explanation": f"The policy '{policy_name}' rule '{rule_description}' (condition: '{condition}') applies. Approval is: {approval_required}. (No specific amount-based filtering applied)"
        }

In [ ]:
print("\n--- Task 3: Apply Rule Filtering Experiment ---")
queries_to_test = [
    "buy device for 800$",
    "buy device for 1800$",
    "buy device for 3500$"
]

for query in queries_to_test:
    print(f"\nUser Query: {query}")
    # Step 1: LLM Re-Ranking (using semantic candidates first)
    semantic_candidates = retrive_cand(query, policy_embeddings)
    if semantic_candidates:
        llm_selected_rule_text = llm_rerank_candidates(query, semantic_candidates)
        print(f"LLM Selected Rule: {llm_selected_rule_text}")

        # Step 2: Apply Rule Filtering
        filter_result = apply_rule_filtering(query, llm_selected_rule_text)
        if filter_result:
            print(f"Rule Filtering Result: {filter_result['is_match']}")
            print(f"Explanation: {filter_result['explanation']}")
        else:
            print("Rule Filtering Result: No matching policy found or condition could not be evaluated.")
    else:
        print("No candidate policies found for LLM re-ranking.")


--- Task 3: Apply Rule Filtering Experiment ---

User Query: buy device for 800$
LLM Selected Rule: Financial | Hardware Purchase Policy | Purchases up to $1500 are auto-approved | 1500.0 | amount <= 1500 | No | Medium
Rule Filtering Result: True
Explanation: The policy 'Hardware Purchase Policy' rule 'Purchases up to $1500 are auto-approved' applies because the amount $800 meets the condition 'amount <= 1500'. Approval is: No.

User Query: buy device for 1800$
LLM Selected Rule: Financial | Hardware Purchase Policy | Purchases between $1501–$3000 require manager approval and 2 quotes | 3000.0 | 1500 < amount <= 3000 | Manager | Medium
Rule Filtering Result: True
Explanation: The policy 'Hardware Purchase Policy' rule 'Purchases between $1501–$3000 require manager approval and 2 quotes' applies because the amount $1800 meets the condition '1500 < amount <= 3000'. Approval is: Manager.

User Query: buy device for 3500$
LLM Selected Rule: Financial | Hardware Purchase Policy | Purchases


## Task 4 — Let the LLM Explain the Decision

Now use the LLM to generate a **short explanation** for the decision.

Input:

* user question
* matched rule

Example

User request

```
buy device for 4000$
```

Rule

```
amount > 3000
```

Example explanation

```
The requested amount is higher than the policy limit of 3000$, therefore higher approval is required.
```

### Experiment

Try multiple queries and observe how the explanation changes.

---

In [ ]:
def generate_decision_explanation(user_query, matched_policy_details):
    """
    Uses the LLM to generate a short explanation for the decision.

    Args:
        user_query (str): The original user's question.
        matched_policy_details (dict): A dictionary containing details of the matched policy,
                                       including 'policy_name', 'rule_description', 'condition', and 'approval_required'.

    Returns:
        str: A natural language explanation of the decision.
    """
    policy_name = matched_policy_details['policy_name']
    rule_description = matched_policy_details['rule_description']
    condition = matched_policy_details['condition']
    approval_required = matched_policy_details['approval_required']

    prompt_content = f"""User Question: {user_query}
Matched Policy:
  Policy Name: {policy_name}
  Rule Description: {rule_description}
  Condition: {condition}
  Approval Required: {approval_required}

Based on the matched policy, provide a short, clear explanation of the decision. 
Focus on what the user needs to do or understand based on their request and the policy rule. 
Your explanation MUST be derived SOLELY from the 'Matched Policy' details provided above. 
Do not invent or assume any information not explicitly present in the matched policy. 
Start directly with the explanation without prefacing it with 'According to the policy' or similar phrases.

Explanation:"""

    try:
        response = client.chat.completions.create(
            model="openai/gpt-3.5-turbo", # Using an OpenRouter compatible model
            messages=[
                {"role": "user", "content": prompt_content}
            ],
            temperature=0.0
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        return f"Error generating explanation with LLM: {e}"


In [55]:
print("\n--- Task 4: LLM Explains Decision Experiment ---")
queries_to_test = [
    "buy device for 800$",
    "buy device for 1800$",
    "buy device for 3500$"
]

for query in queries_to_test:
    print(f"\nUser Query: {query}")
    # Step 1: LLM Re-Ranking (using semantic candidates first)
    semantic_candidates = retrive_cand(query, policy_embeddings)
    if semantic_candidates:
        llm_selected_rule_text = llm_rerank_candidates(query, semantic_candidates)
        # print(f"LLM Selected Rule: {llm_selected_rule_text}")

        # Step 2: Apply Rule Filtering
        filter_result = apply_rule_filtering(query, llm_selected_rule_text)

        if filter_result and filter_result['is_match']:
            # Step 3: LLM Explanation (Task 4)
            explanation = generate_decision_explanation(query, filter_result)
            print(f"Decision Explanation: {explanation}")
        elif filter_result:
            print(f"Decision: {filter_result['explanation']}")
        else:
            print("Decision: No matching policy found or condition could not be evaluated.")
    else:
        print("Decision: No candidate policies found for LLM re-ranking.")


--- Task 4: LLM Explains Decision Experiment ---

User Query: buy device for 800$
Decision Explanation: Your purchase of a device for $800 falls within the approved limit of $1500 stated in the Hardware Purchase Policy. No further approval is required for this transaction.

User Query: buy device for 1800$
Decision Explanation: Since the device you want to purchase is priced at $1800, it falls within the range of $1501-$3000, which requires manager approval and obtaining 2 quotes. Therefore, you will need to seek approval from your manager before proceeding with the purchase.

User Query: buy device for 3500$
Decision Explanation: Since the device you are looking to purchase is priced at $3500, it falls under the Hardware Purchase Policy rule that requires approval from the Manager, Finance, and CTO. Before proceeding with the purchase, you will need to obtain approval from all three individuals mentioned.







## Task 5 — LLM Guardrail Task

Sometimes LLMs may generate answers that **do not follow the policy rules**.

In real systems we must prevent this.

Your task is to build a **Guardrail** that ensures the LLM only answers using valid policies.

### Goal

Create a validation step before returning the LLM answer.

The guardrail should check:

* Is the answer based on a retrieved policy?
* Does the rule condition match the user request?

If not, the system should **reject the response**.

### Example

User request

```
Can I buy a device for 10000$ without approval?
```

The LLM might try to answer freely.

The guardrail should force the system to respond using the correct policy.

Example safe response

```
According to the Hardware Purchase Policy, purchases above $3000 require manager, finance, and CTO approval.
```

### Experiment

Try questions that might confuse the model:

```
Can I skip approval for a 5000$ purchase?
Is there a way to bypass the purchase limit?
Can I buy hardware without following policy?
```

Check if your guardrail forces the system to follow the correct rule.

---


In [57]:
def llm_guardrail_decision(user_query):
    # Step 1: Hybrid Retrieval
    candidate_policies = hybrid_retrieve(user_query, policy_embeddings, policy_texts, bm25)
    if not candidate_policies:
        return "Guardrail Triggered: No relevant policies found for your request. Please try rephrasing."

    # Step 2: LLM Re-Ranking
    llm_selected_rule_text = llm_rerank_candidates(user_query, candidate_policies)
    if llm_selected_rule_text is None:
        return "Guardrail Triggered: The AI could not confidently select a policy from the candidates. Please rephrase your question."

    # Step 3: Rule Filtering (Validation Step for Guardrail)
    filter_result = apply_rule_filtering(user_query, llm_selected_rule_text)

    if filter_result and filter_result['is_match']:
        # Guardrail passed - proceed to explanation
        explanation = generate_decision_explanation(user_query, filter_result)
        return f"Decision: {explanation}"
    elif filter_result:
        # Guardrail Triggered: Rule condition mismatch
        return f"Guardrail Triggered: The policy found does not apply to your specific request. {filter_result['explanation']}"
    else:
        # Guardrail Triggered: General filtering failure
        return "Guardrail Triggered: Could not validate the policy against your request."

In [ ]:
# Experiment with questions that might confuse the model
print("\n--- Task 5: LLM Guardrail Experiment ---")
guardrail_queries = [
    "Can I skip approval for a 5000$ purchase?",
    "Is there a way to bypass the purchase limit?",
    "Can I buy hardware without following policy?"
]

for query in guardrail_queries:
    print(f"\nUser Query: {query}")
    decision = llm_guardrail_decision(query)
    print(decision)



--- Task 5: LLM Guardrail Experiment ---

User Query: Can I skip approval for a 5000$ purchase?
Decision: You cannot skip approval for a $5000 purchase as purchases above $3000 require manager, finance, and CTO approval.

User Query: Is there a way to bypass the purchase limit?
Guardrail Triggered: The policy found does not apply to your specific request. No amount found in query to evaluate against condition 'amount > 3000'.

User Query: Can I buy hardware without following policy?
Guardrail Triggered: The policy found does not apply to your specific request. No amount found in query to evaluate against condition 'amount > 3000'.


In [59]:
# Also test with a valid query to ensure it still works
print("\n--- Task 5: Guardrail Test with Valid Query ---")
valid_query = "I need to buy a device for 3500$"
print(f"\nUser Query: {valid_query}")
decision = llm_guardrail_decision(valid_query)
print(decision)



--- Task 5: Guardrail Test with Valid Query ---

User Query: I need to buy a device for 3500$
Decision: To purchase a device for $3500, you will need to obtain approval from the manager, finance department, and the Chief Technology Officer (CTO) as per the Hardware Purchase Policy. Make sure to follow the necessary steps to get the required approvals before proceeding with the purchase.


## Task 6 — Build the Final AI Decision System

Now combine everything into a complete pipeline.

Final pipeline:

```
User Question
↓
Hybrid Retrieval
↓
Candidate Policies
↓
LLM Re‑Ranking
↓
Rule Filtering
↓
LLM Explanation
```

Create a single function that runs the entire pipeline.

### Experiment

Run the system with different questions:

```
I need to buy device for 800$
I need to buy device for 1800$
I need to buy device for 3500$
I need a flight for 4 hours
I need a flight for 12 hours
```
Observe how the decision changes.

---


In [60]:
import logging

# Configure basic logging (using print for simplicity as full logging setup is not provided)
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

def run_decision_system(question: str) -> dict:
    logging.info(f"Starting decision system for question: '{question}'")

    result = {
        "question": question,
        "retrieved_docs": [],
        "candidate_policies": [],
        "reranked_candidates": [],
        "filtered_candidates": [],
        "final_decision": "No decision could be made.",
        "explanation": "No relevant policies were found or conditions were not met."
    }

    # Step 1 — Hybrid Retrieval
    logging.info("Step 1: Performing Hybrid Retrieval...")
    retrieved_docs = hybrid_retrieve(question, policy_embeddings, policy_texts, bm25)
    result["retrieved_docs"] = retrieved_docs
    result["candidate_policies"] = retrieved_docs # In this pipeline, retrieved docs are the candidates

    if not retrieved_docs:
        result["final_decision"] = "No relevant policies found."
        result["explanation"] = "Hybrid retrieval did not find any policies matching the query."
        logging.warning("No policies retrieved. Exiting pipeline.")
        return result

    # Step 2 — LLM Re-Ranking
    logging.info("Step 2: Performing LLM Re-Ranking...")
    llm_selected_rule_text = llm_rerank_candidates(question, retrieved_docs)

    if llm_selected_rule_text:
        result["reranked_candidates"].append(llm_selected_rule_text)
        logging.info(f"LLM re-ranked and selected: '{llm_selected_rule_text}'")
    else:
        result["final_decision"] = "LLM could not confidently re-rank candidates."
        result["explanation"] = "The LLM was unable to select a single most relevant policy from the retrieved candidates."
        logging.warning("LLM re-ranking failed. Exiting pipeline.")
        return result

    # Step 3 — Rule Filtering
    logging.info("Step 3: Applying Rule Filtering...")
    filter_result = apply_rule_filtering(question, llm_selected_rule_text)

    if filter_result and filter_result['is_match']:
        result["filtered_candidates"].append(filter_result)
        result["final_decision"] = f"Policy applied: {filter_result['policy_name']} - {filter_result['rule_description']}"
        logging.info(f"Rule filtering successful: '{filter_result['explanation']}'")

        # Step 4 — LLM Explanation
        logging.info("Step 4: Generating LLM Explanation...")
        explanation_text = generate_decision_explanation(question, filter_result)
        result["explanation"] = explanation_text
        logging.info("LLM explanation generated.")
    else:
        result["final_decision"] = "No policy condition met after filtering."
        result["explanation"] = filter_result.get('explanation', "The LLM-selected policy's conditions were not met by the user's request, or an error occurred during evaluation.")
        logging.warning(f"Rule filtering failed or condition not met: {result['explanation']}. Exiting pipeline.")
        return result

    logging.info("Decision system pipeline completed successfully.")
    return result

In [61]:
# Experiment with test queries
print("\n--- Running Final AI Decision System ---")
test_queries = [
    "I need to buy device for 800$",
    "I need to buy device for 1800$",
    "I need to buy device for 3500$",
    "I need a flight for 4 hours",
    "I need a flight for 12 hours"
]

for query in test_queries:
    print("\n" + "="*80)
    print(f"Processing User Query: {query}")
    print("="*80)
    decision_output = run_decision_system(query)

    print("\n------------------------------------------------")
    print(f"User Question: {decision_output['question']}")
    print(f"Retrieved Policies ({len(decision_output['retrieved_docs'])}):")
    for doc in decision_output['retrieved_docs']:
        print(f"  - Score: {doc['Score']:.3f}, Text: {doc['full_text']}")
    print(f"Top Ranked Policies ({len(decision_output['reranked_candidates'])}):")
    for cand in decision_output['reranked_candidates']:
        print(f"  - {cand}")
    print(f"Filtered Policies ({len(decision_output['filtered_candidates'])}):")
    for filt in decision_output['filtered_candidates']:
        print(f"  - Match: {filt.get('is_match')}, Policy: {filt.get('policy_name')}, Rule: {filt.get('rule_description')}")
    print(f"Final Decision: {decision_output['final_decision']}")
    print(f"Explanation: {decision_output['explanation']}")
    print("------------------------------------------------")



--- Running Final AI Decision System ---

Processing User Query: I need to buy device for 800$

------------------------------------------------
User Question: I need to buy device for 800$
Retrieved Policies (5):
  - Score: 1.000, Text: Financial | Hardware Purchase Policy | Purchases between $1501–$3000 require manager approval and 2 quotes | 3000.0 | 1500 < amount <= 3000 | Manager | Medium
  - Score: 0.821, Text: Financial | Hardware Purchase Policy | Purchases up to $1500 are auto-approved | 1500.0 | amount <= 1500 | No | Medium
  - Score: 0.649, Text: Financial | Hardware Purchase Policy | Purchases above $3000 require manager, finance, and CTO approval | 3000.0 | amount > 3000 | Manager+Finance+CTO | Medium
  - Score: 0.429, Text: Financial | Hardware Purchase Policy | Annual hardware budget per employee is $2500 | 2500.0 | annual_budget_limit | No | Medium
  - Score: 0.250, Text: Financial | Hardware Purchase Policy | Vendor must be selected from approved vendor list | nan | v

In [62]:
import ipywidgets as widgets
from IPython.display import display, HTML

# Create a Textarea for the user's question
question_input = widgets.Textarea(
    value='',
    placeholder='Enter your question here (e.g., I need to buy device for 800$)',
    description='Question:',
    disabled=False,
    layout=widgets.Layout(width='auto', height='80px')
)

# Create a Button to trigger the decision system
run_button = widgets.Button(
    description='Get Decision',
    disabled=False,
    button_style='info', # 'success', 'info', 'warning', 'danger' or ''
    tooltip='Click to get the AI decision',
    icon='play'
)

# Create an Output widget to display results
output_area = widgets.Output()

# Function to handle button click
def on_button_click(b):
    with output_area:
        output_area.clear_output() # Clear previous output
        user_question = question_input.value.strip()
        if not user_question:
            print("Please enter a question.")
            return

        print(f"Processing User Query: {user_question}\n")
        decision_output = run_decision_system(user_question)

        # Display the structured output
        print("------------------------------------------------")
        print(f"User Question: {decision_output['question']}")
        print(f"Retrieved Policies ({len(decision_output['retrieved_docs'])}):")
        for doc in decision_output['retrieved_docs']:
            print(f"  - Score: {doc['Score']:.3f}, Text: {doc['full_text']}")
        print(f"Top Ranked Policies ({len(decision_output['reranked_candidates'])}):")
        for cand in decision_output['reranked_candidates']:
            print(f"  - {cand}")
        print(f"Filtered Policies ({len(decision_output['filtered_candidates'])}):")
        for filt in decision_output['filtered_candidates']:
            print(f"  - Match: {filt.get('is_match')}, Policy: {filt.get('policy_name')}, Rule: {filt.get('rule_description')}")
        print(f"Final Decision: {decision_output['final_decision']}")
        print(f"Explanation: {decision_output['explanation']}")
        print("------------------------------------------------")

# Attach the button click event to the handler function
run_button.on_click(on_button_click)

# Display the widgets
display(question_input, run_button, output_area)


Textarea(value='', description='Question:', layout=Layout(height='80px', width='auto'), placeholder='Enter you…

Button(button_style='info', description='Get Decision', icon='play', style=ButtonStyle(), tooltip='Click to ge…

Output()

In [63]:
def get_ai_decision(user_query):
    # Step 1: Hybrid Retrieval
    candidate_policies = hybrid_retrieve(user_query, policy_embeddings, policy_texts, bm25)
    if not candidate_policies:
        return "Decision: No relevant policies found for your request. Please try rephrasing."

    # Step 2: LLM Re-Ranking
    llm_selected_rule_text = llm_rerank_candidates(user_query, candidate_policies)
    if llm_selected_rule_text is None:
        return "Decision: The AI could not confidently select a policy from the candidates. Please rephrase your question."

    # Step 3: Rule Filtering
    filter_result = apply_rule_filtering(user_query, llm_selected_rule_text)

    if filter_result and filter_result['is_match']:
        # Step 4: LLM Explanation
        explanation = generate_decision_explanation(user_query, filter_result)
        return f"Decision: {explanation}"
    elif filter_result:
        return f"Decision: The policy found does not apply to your specific request. {filter_result['explanation']}"
    else:
        return "Decision: Could not validate the policy against your request."


In [64]:
print("\n--- Task 6: Final AI Decision System Experiment ---")
queries_to_test = [
    "I need to buy device for 800$",
    "I need to buy device for 1800$",
    "I need to buy device for 3500$",
    "I need a flight for 4 hours",
    "I need a flight for 12 hours"
]

for query in queries_to_test:
    print(f"\nUser Query: {query}")
    decision = get_ai_decision(query)
    print(decision)



--- Task 6: Final AI Decision System Experiment ---

User Query: I need to buy device for 800$
Decision: Your purchase of a device for $800 falls within the approved limit of $1500 under the Hardware Purchase Policy. No approval is required for this purchase. You can proceed with buying the device without any further steps needed.

User Query: I need to buy device for 1800$
Decision: Since your device purchase falls within the $1501-$3000 range, you will need to obtain manager approval and provide two quotes before proceeding with the purchase. Make sure to follow this process to ensure compliance with the Hardware Purchase Policy.

User Query: I need to buy device for 3500$
Decision: To purchase a device for $3500, you will need to obtain approval from the manager, finance department, and the Chief Technology Officer (CTO) as per the Hardware Purchase Policy. Make sure to follow the necessary steps to get the required approvals before proceeding with the purchase.

User Query: I need